# Sri Sivasubramaniya Nadar College of Engineering, Chennai
### (An Autonomous Institution Affiliated to Anna University)
**Degree & Branch:** M. Tech (Integrated) Computer Science & Engineering | **Semester:** V
**Subject Code & Name:** ICS1512 & Machine Learning Algorithms Laboratory
**Academic Year:** 2025-2026 (Odd) | **Batch:** 2024-2029

---
## Experiment 8: Clustering Human Activity Recognition Data using K-Means, DBSCAN, and Hierarchical Clustering

**Student Name:** Danusu K | **Register Number:** 3122247001013
**Faculty:** Dr. Poreddy Ajay Kumar Reddy

### Objectives:
1. Implement and analyze **K-Means**, **DBSCAN**, and **Hierarchical Agglomerative Clustering (HAC)** on the UCI Human Activity Recognition (HAR) Using Smartphones dataset.
2. Select the optimal number of clusters for K-Means using the **Elbow Method** and **Silhouette Score**.
3. Tune DBSCAN's $\epsilon$ (neighborhood radius) and minPts via the **k-distance graph** and grid search.
4. Compare **single / complete / average / Ward** linkage criteria for Hierarchical Clustering and visualize dendrograms.
5. Visualize clusters in 2D using **PCA** and **t-SNE**, and evaluate against ground-truth activity labels using internal (Silhouette, Davies-Bouldin, Calinski-Harabasz) and external (ARI, NMI) metrics.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from scipy.cluster.hierarchy import dendrogram, linkage

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment initialized successfully with seed = 42.")

## 2. Dataset Ingestion & Preprocessing
Load the UCI HAR Dataset's pre-extracted 561-feature train and test partitions (30 volunteers, 6 activities, 128-sample / 2.56s sliding windows), merge them into a single unsupervised clustering corpus, and standardize.

In [ ]:
DATA_DIR = "../dataset"
ACTIVITY_NAMES = {1: "WALKING", 2: "WALKING_UPSTAIRS", 3: "WALKING_DOWNSTAIRS",
                   4: "SITTING", 5: "STANDING", 6: "LAYING"}

X_train = np.loadtxt(os.path.join(DATA_DIR, "train", "X_train.txt"))
y_train = np.loadtxt(os.path.join(DATA_DIR, "train", "y_train.txt")).astype(int)
subj_train = np.loadtxt(os.path.join(DATA_DIR, "train", "subject_train.txt")).astype(int)

X_test = np.loadtxt(os.path.join(DATA_DIR, "test", "X_test.txt"))
y_test = np.loadtxt(os.path.join(DATA_DIR, "test", "y_test.txt")).astype(int)
subj_test = np.loadtxt(os.path.join(DATA_DIR, "test", "subject_test.txt")).astype(int)

X = np.vstack([X_train, X_test])
y = np.concatenate([y_train, y_test])
subjects = np.concatenate([subj_train, subj_test])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Merged dataset shape: {X.shape} | Missing values: {np.isnan(X).sum()}")
print(f"Subjects: {len(np.unique(subjects))} | Activities: {len(np.unique(y))}")

## 3. Principal Component Analysis: Scree Analysis & Dimensionality Reduction
The 561 time/frequency-domain HAR features are heavily correlated (many are derived from the same tri-axial signals). PCA is used to (a) visualize the manifold in 2D, and (b) reduce dimensionality to **90% cumulative variance** before clustering — mitigating the curse of dimensionality for the Euclidean-distance-based K-Means, DBSCAN, and HAC algorithms.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

n_selected = int(np.argmax(cum_var >= 0.90) + 1)
pca_reduced = PCA(n_components=n_selected, random_state=RANDOM_STATE)
X_pca = pca_reduced.fit_transform(X_scaled)

print(f"Total standardized features: {X_scaled.shape[1]}")
print(f"Components retained for 90% variance: {n_selected} ({cum_var[n_selected-1]*100:.2f}%)")
print(f"Dimensionality reduction: {(1 - n_selected/X_scaled.shape[1])*100:.1f}%")

## 4. Model A — K-Means: Elbow Method & Silhouette Scan
Sweep $k = 2 \ldots 8$, recording WCSS (inertia), Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index, and external agreement with ground-truth activity labels (ARI, NMI).

In [ ]:
kmeans_records = []
kmeans_models = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels, sample_size=5000, random_state=RANDOM_STATE)
    kmeans_records.append({
        "k": k, "wcss": km.inertia_, "silhouette": sil,
        "ari": adjusted_rand_score(y, labels), "nmi": normalized_mutual_info_score(y, labels)
    })
    kmeans_models[k] = km

df_kmeans = pd.DataFrame(kmeans_records)
best_k = int(df_kmeans.loc[df_kmeans["silhouette"].idxmax(), "k"])
print(f"Best k by Silhouette Score: {best_k}")
df_kmeans

## 5. Model B — DBSCAN: k-Distance Graph & (eps, minPts) Grid Search
The 10-NN distance graph guides candidate $\epsilon$ values (knee-region percentiles); minPts is swept over $\{5, 10, 15, 20\}$. The configuration maximizing Silhouette Score (excluding noise, with noise ratio < 50%) is selected.

In [ ]:
nn = NearestNeighbors(n_neighbors=10).fit(X_pca)
distances, _ = nn.kneighbors(X_pca)
k_distances = np.sort(distances[:, -1])

eps_candidates = sorted(set(np.round(np.percentile(k_distances, p), 2) for p in [80, 85, 88, 90, 92, 95]))
print("Candidate eps values (from k-distance percentiles):", eps_candidates)

## 6. Model C — Hierarchical Agglomerative Clustering (Linkage Comparison)
Given the $O(n^2)$ memory/time complexity of agglomerative clustering, a stratified subsample (~2,500 points, proportional to class balance) is used for linkage comparison and dendrogram visualization. **Ward's linkage** is used as the primary configuration per the experiment specification (minimum-variance criterion, most balanced partitions).

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
sub_idx_list = []
for act_id in range(1, 7):
    idx_act = np.where(y == act_id)[0]
    n_take = int(2500 * len(idx_act) / len(y))
    sub_idx_list.append(rng.choice(idx_act, size=n_take, replace=False))
subsample_idx = np.concatenate(sub_idx_list)
rng.shuffle(subsample_idx)

X_sub, y_sub = X_pca[subsample_idx], y[subsample_idx]
print(f"HAC stratified subsample size: {len(subsample_idx)}")

## 7. Consolidated Results (from `experiment8_results.json`)

In [ ]:
with open("experiment8_results.json", "r") as f:
    results = json.load(f)

print("Dataset:", results["dataset_info"])
print("\nPCA: 90%% variance retained with", results["pca_info"]["chosen_components"], "components")
print("\nK-Means best k:", results["kmeans"]["best_k"])
print("DBSCAN best config:", results["dbscan"]["best_config"])
print("HAC primary linkage:", results["hierarchical"]["best_linkage"])

In [ ]:
kmeans_df = pd.DataFrame(results["kmeans"]["elbow_table"])
kmeans_df

In [ ]:
summary_rows = []
for name, sil, ari, nmi in zip(
    results["metrics_summary"]["algo_names"],
    results["metrics_summary"]["sil_vals"],
    results["metrics_summary"]["ari_vals"],
    results["metrics_summary"]["nmi_vals"]
):
    summary_rows.append({"Algorithm": name, "Silhouette": round(sil, 4), "ARI": round(ari, 4), "NMI": round(nmi, 4)})
pd.DataFrame(summary_rows)

## 8. Visualizations
All plots below are generated by `experiment8.py` and saved to `../output_plots/`.

In [ ]:
from IPython.display import Image, display

plot_files = [
    "../output_plots/01_activity_class_distribution.png",
    "../output_plots/02_pca_scree_and_cumulative_variance.png",
    "../output_plots/03_kmeans_elbow_and_silhouette.png",
    "../output_plots/04_dbscan_kdistance_graph.png",
    "../output_plots/05_dbscan_grid_search_heatmaps.png",
    "../output_plots/06_dendrograms_linkage_comparison.png",
    "../output_plots/07_pca_2d_cluster_scatter.png",
    "../output_plots/08_hac_pca_2d_cluster_scatter.png",
    "../output_plots/09_tsne_cluster_comparison.png",
    "../output_plots/10_internal_metrics_comparison.png",
    "../output_plots/11_external_metrics_comparison.png",
    "../output_plots/12_cluster_activity_contingency.png",
]
for p in plot_files:
    if os.path.exists(p):
        print(f"Displaying: {os.path.basename(p)}")
        display(Image(filename=p, width=800))

## 9. Key Conclusions & Insights
1. **K-Means finds a bimodal structure, not six activities.** Silhouette Score peaks sharply at $k=2$ (0.437) and decays monotonically thereafter; the dominant axis of variation (PC1, 50.7% variance) separates **dynamic** activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS) from **static** activities (SITTING, STANDING, LAYING) almost perfectly, while SITTING/STANDING are nearly inseparable in feature space.
2. **k = 6 (matching true activity count) is not the internally optimal k**, but yields the best external agreement among K-Means candidates (ARI = 0.4201, NMI = 0.560 at k=6-8) — illustrating the classic tension between unsupervised internal validity and external ground-truth alignment.
3. **DBSCAN struggles to recover activity structure.** Even after PCA (90% variance, 65D) and careful $\epsilon$/minPts tuning, DBSCAN only ever recovers the same coarse dynamic-vs-static split as K-Means (best config: 2 clusters, 2.2% noise, Silhouette = 0.409) but with far lower external agreement (ARI = 0.003) — density-based clustering is not well-suited to this continuous, overlapping sensor manifold.
4. **Linkage choice drastically changes Hierarchical Clustering behavior.** Single and Average linkage produce degenerate chaining (one dominant cluster absorbing >99% of points, near-zero ARI) despite high raw Silhouette Scores. Ward's linkage produces the most balanced, activity-aligned partition (ARI = 0.305, NMI = 0.476), consistent with its minimum-variance objective.
5. **Best overall external agreement:** K-Means ($k=2$, ARI = 0.330, NMI = 0.545) and Ward HAC (ARI = 0.305, NMI = 0.476) both substantially outperform DBSCAN, confirming that centroid- and variance-based partitioning methods align better with the HAR activity manifold than density-based or chaining-prone linkage methods.